# Lab 1E: Data Modeling in Python

**Time**: ~60 min  
**Environment**: Jupyter kernel in VS Code

In this exercise you compare two data models for the same e-commerce domain — a **reference (normalized)** model that mirrors a relational schema, and an **embed (denormalized)** model designed around the queries the app actually runs. You then revisit partition-key choice with a hot-partition / composite-key demo on a separate provisioned account.

## Prerequisites

- Python 3.10+ with `azure-cosmos`, `azure-identity`, and `python-dotenv` installed: `pip install azure-cosmos azure-identity python-dotenv`
- `COSMOS_ENDPOINT` environment variable set to your Cosmos DB account endpoint
- `COSMOS_ENDPOINT_PROVISIONED` environment variable set to your Cosmos DB provisioned throughput account endpoint

Run each cell in order. Most cells are prebuilt — three have **student exercises** marked in the code (Steps 2, 4, and 6).

## Step 0: Setup — Seed reference and embed databases

Connect to the **serverless** Cosmos account and seed `ModelingReference` and `ModelingEmbed` from `seed-reference.json` and `seed-embed.json`. Every document writes the same fixed `partitionKey = "default"` — the dataset is small enough to live in one logical partition, so partitioning isn't needed for scale and queries stay simple.

Steps 5–7 also touch a **provisioned** account (different endpoint, `COSMOS_ENDPOINT_PROVISIONED`) — that client is created lazily when those steps run.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import json
import os
from datetime import datetime, timezone
from azure.cosmos import CosmosClient, PartitionKey
from azure.cosmos.exceptions import CosmosResourceNotFoundError
from azure.identity import AzureCliCredential

ENDPOINT = os.environ.get("COSMOS_ENDPOINT")
if not ENDPOINT:
    raise RuntimeError("COSMOS_ENDPOINT environment variable is required (see SetEnv.ps1).")

REF_DB_NAME = "ModelingReference"
EMBED_DB_NAME = "ModelingEmbed"
PROVISIONED_DB_NAME = "Modeling"

PK_VALUE = "default"
PK = PartitionKey(PK_VALUE)

cred = AzureCliCredential()
client = CosmosClient(url=ENDPOINT, credential=cred)
ref_db = client.get_database_client(REF_DB_NAME)
embed_db = client.get_database_client(EMBED_DB_NAME)

# Lazily created when steps 5–7 run.
provisioned_client = None
provisioned_db = None

def ensure_provisioned_client():
    global provisioned_client, provisioned_db
    if provisioned_client is not None:
        return provisioned_db
    endpoint = os.environ.get("COSMOS_ENDPOINT_PROVISIONED")
    if not endpoint:
        raise RuntimeError(
            "Steps 5–7 require COSMOS_ENDPOINT_PROVISIONED."
        )
    provisioned_client = CosmosClient(url=endpoint, credential=cred)
    provisioned_db = provisioned_client.get_database_client(PROVISIONED_DB_NAME)
    print(f"  provisioned endpoint: {endpoint}")
    print(f"  modeling DB:          {PROVISIONED_DB_NAME}")
    return provisioned_db

print(f"  serverless endpoint: {ENDPOINT}")
print(f"  reference DB:        {REF_DB_NAME}")
print(f"  embed DB:            {EMBED_DB_NAME}")

In [ ]:
with open("seed-reference.json", "r", encoding="utf-8") as f:
    ref_seed = json.load(f)
with open("seed-embed.json", "r", encoding="utf-8") as f:
    embed_seed = json.load(f)

def upsert_all(container, items):
    for item in items:
        container.upsert_item(body=item)
        print(f"    upserted {container.id}/{item['id']}")
    print(f"  -> {len(items)} docs in {container.id}")

print(f"Seeding '{REF_DB_NAME}' (one container per entity) ...")
upsert_all(ref_db.get_container_client("Customers"),         ref_seed["customers"])
upsert_all(ref_db.get_container_client("Addresses"),         ref_seed["addresses"])
upsert_all(ref_db.get_container_client("ProductCategories"), ref_seed["productCategories"])
upsert_all(ref_db.get_container_client("Products"),          ref_seed["products"])
upsert_all(ref_db.get_container_client("Orders"),            ref_seed["orders"])
upsert_all(ref_db.get_container_client("OrderItems"),        ref_seed["orderItems"])

print(f"\nSeeding '{EMBED_DB_NAME}' (denormalized — addresses on customer, lines on order) ...")
upsert_all(embed_db.get_container_client("Customers"), embed_seed["customers"])
upsert_all(embed_db.get_container_client("Products"),  embed_seed["products"])
upsert_all(embed_db.get_container_client("Orders"),    embed_seed["orders"])

## Step 1: Fetch a complete order — REFERENCE model

Walking references across six containers — `Orders → OrderItems → Products → ProductCategories`, plus `Customers` and `Addresses`. Each hop is its own round-trip and its own RU charge. In a relational DB the equivalent would be a single SQL JOIN; in Cosmos every query is single-container, so we do it by hand and tally the cost.

**Expected behavior**: 7+ round-trips; the RU charge sums across all of them.

In [ ]:
def last_ru(container):
    return float(container.client_connection.last_response_headers["x-ms-request-charge"])

target_order_id = "order_001"
total_ru = 0.0
round_trips = 0

orders     = ref_db.get_container_client("Orders")
order_items= ref_db.get_container_client("OrderItems")
products   = ref_db.get_container_client("Products")
categories = ref_db.get_container_client("ProductCategories")
customers  = ref_db.get_container_client("Customers")
addresses  = ref_db.get_container_client("Addresses")

order = orders.read_item(item=target_order_id, partition_key=PK_VALUE)
ru = last_ru(orders); total_ru += ru; round_trips += 1
print(f"  [1] Read Orders/{order['id']:<10}                              {ru:5.2f} RU")

items = list(order_items.query_items(
    query="SELECT * FROM c WHERE c.orderId = @orderId",
    parameters=[{"name": "@orderId", "value": order["id"]}],
    partition_key=PK_VALUE,
))
ru = last_ru(order_items); total_ru += ru; round_trips += 1
print(f"  [2] Query OrderItems WHERE orderId='{order['id']}'    {ru:5.2f} RU ({len(items)} rows)")

for item in items:
    product = products.read_item(item=item["productId"], partition_key=PK_VALUE)
    ru = last_ru(products); total_ru += ru; round_trips += 1
    print(f"  [3] Read Products/{item['productId']:<10}                          {ru:5.2f} RU  ({product['name']})")

    category = categories.read_item(item=product["categoryId"], partition_key=PK_VALUE)
    ru = last_ru(categories); total_ru += ru; round_trips += 1
    print(f"  [4] Read ProductCategories/{product['categoryId']:<10}              {ru:5.2f} RU  ({category['category']})")

customer = customers.read_item(item=order["customerId"], partition_key=PK_VALUE)
ru = last_ru(customers); total_ru += ru; round_trips += 1
print(f"  [5] Read Customers/{order['customerId']:<10}                         {ru:5.2f} RU  ({customer['name']})")

address = addresses.read_item(item=order["addressId"], partition_key=PK_VALUE)
ru = last_ru(addresses); total_ru += ru; round_trips += 1
print(f"  [6] Read Addresses/{order['addressId']:<10}                          {ru:5.2f} RU  ({address['street']}, {address['city']})")

print(f"\n  Totals: {round_trips} round-trips, {total_ru:.2f} RU")

## Step 2: Fetch a complete order — EMBED model (STUDENT EXERCISE)

The customer snapshot and line items already live on the order document, so a single point read returns the whole order. The contrast with Step 1's six-container walk is the whole point of the embed model — write it yourself to see how little code it takes.

Replace the placeholder `order` dict in the code cell with a single `read_item` call on the embed `Orders` container, and capture the RU charge:

```python
order = embed_orders.read_item(item="order_001", partition_key=PK_VALUE)
ru = last_ru(embed_orders)
```

**Expected output**: 1 round-trip, ~1–3 RU. Compare directly to Step 1's totals.

In [ ]:
embed_orders = embed_db.get_container_client("Orders")

# STUDENT EXERCISE: replace the placeholder below with a read_item call for
# order_001 in the embed Orders container, and capture the RU charge.
# See the markdown cell above for the snippet.
order = {
    "id": "(placeholder)",
    "customerName": "(placeholder)", "customerId": "(placeholder)",
    "customerStreet": "(placeholder)", "customerCity": "(placeholder)",
    "customerState": "(placeholder)", "customerZipCode": "(placeholder)",
    "date": "0000-00-00",
    "totalAmount": 0.0,
    "lines": [],
}
ru = 0.0

print(f"  Read Orders/{order['id']}                              {ru:5.2f} RU")
print(f"    customer:   {order['customerName']} <{order['customerId']}>")
print(f"    ship to:    {order['customerStreet']}, {order['customerCity']}, {order['customerState']} {order['customerZipCode']}")
print(f"    date/total: {order['date'][:10]}   ${order['totalAmount']:.2f}")
print(f"    lines:")
for line in order["lines"]:
    print(f"      - {line['quantity']} x {line['productName']:<22} ({line['productCategory']:<12}) @ ${line['productPrice']:6.2f} = ${line['lineTotal']:7.2f}")

print(f"\n  Totals: 1 round-trip, {ru:.2f} RU")

## Step 3: Update a customer address — model tradeoffs

cust_001 moves from `100 Main St, Seattle WA` to `555 New Lane, Bellevue WA`.

- **Reference model**: update the single `Addresses` row; every order resolves the new address on its next read. **1 write.**
- **Embed model**: update the customer doc — but past orders still carry the snapshotted old address. Two valid answers:
  1. **Historical accuracy**: leave past orders alone (they reflect where the order was actually shipped).
  2. **Propagate the change**: fan out an update to every affected order. In production this is the **Cosmos change feed** pattern; the lab does the same work inline with a query + replace so the cost is observable.

In [ ]:
customer_id = "cust_001"
address_id = "addr_001"
new_street, new_city, new_state, new_zip = "555 New Lane", "Bellevue", "WA", "98004"

print("Reference model — update the single Addresses row; every order resolves it on next read.")
ref_addresses = ref_db.get_container_client("Addresses")
addr = ref_addresses.read_item(item=address_id, partition_key=PK_VALUE)
addr["street"] = new_street
addr["city"] = new_city
addr["state"] = new_state
addr["zipCode"] = new_zip
ref_addresses.replace_item(item=address_id, body=addr)
ru = last_ru(ref_addresses)
print(f"  Replaced Addresses/{address_id}    {ru:.2f} RU  (1 write)")

print("\nEmbed model — update the customer doc, but past orders still carry the old snapshotted address.")
embed_customers = embed_db.get_container_client("Customers")
cust = embed_customers.read_item(item=customer_id, partition_key=PK_VALUE)
cust["addresses"][0]["street"] = new_street
cust["addresses"][0]["city"] = new_city
cust["addresses"][0]["state"] = new_state
cust["addresses"][0]["zipCode"] = new_zip
embed_customers.replace_item(item=customer_id, body=cust)
ru = last_ru(embed_customers)
print(f"  Replaced Customers/{customer_id}    {ru:.2f} RU")

stale = embed_orders.read_item(item="order_001", partition_key=PK_VALUE)
print(f"  order_001 ship-to (still snapshotted): {stale['customerStreet']}, {stale['customerCity']}")

print("\nFanning out the update — query affected orders, replace each one (the change-feed pattern, inline).")
affected = list(embed_orders.query_items(
    query="SELECT * FROM c WHERE c.docType = 'order' AND c.customerId = @custId",
    parameters=[{"name": "@custId", "value": customer_id}],
    partition_key=PK_VALUE,
))
fan_out_ru = last_ru(embed_orders)
for ord_doc in affected:
    ord_doc["customerStreet"] = new_street
    ord_doc["customerCity"] = new_city
    ord_doc["customerState"] = new_state
    ord_doc["customerZipCode"] = new_zip
    embed_orders.replace_item(item=ord_doc["id"], body=ord_doc)
    fan_out_ru += last_ru(embed_orders)

print(f"  Fan-out replaced {len(affected)} order doc(s)    {fan_out_ru:.2f} RU")

## Step 4: Designing by usage patterns — orders by customer name (STUDENT EXERCISE)

The snapshotted `customerName` on each order doc makes "find every order for this customer" a single-container query — no cross-container join, no second round-trip to look up the customer first.

**Your job**: replace the placeholder query string below with a parameterized query that returns order docs (those with `docType = 'order'`) for the given `customerName`:

```python
query = "SELECT * FROM c WHERE c.docType = 'order' AND c.customerName = @name"
```

**Expected output**: two orders for `Alice Anderson` printed with their dates and totals, plus the RU charged.

Other patterns this model is shaped for (try them on your own):

| Pattern                       | How it's served                                                                                       |
|-------------------------------|-------------------------------------------------------------------------------------------------------|
| Customer by id                | Point read on `Customers/{id}`. Cheapest possible — typically ~1 RU.                                  |
| Customer by name              | `SELECT * FROM c WHERE c.name = @name` on `Customers`. Single PK so no fan-out.                       |
| Order by id                   | Point read on `Orders/{id}`. The full order — customer snapshot + line items — comes back in one hop. |
| Orders by customer name       | The example this step runs.                                                                           |
| Product list                  | `SELECT * FROM c` on `Products`. Tiny catalog, single PK, no `WHERE` clause needed.                   |
| Revenue by date / month       | `Orders` holds both `OrderDocument` and `ServiceInvoice` (discriminated by `docType`). A single `GROUP BY` on `c.date` rolls up product sales *and* service revenue. |

In [ ]:
customer_name = "Alice Anderson"

# STUDENT EXERCISE: replace the placeholder query below with a parameterized query
# that returns order docs (docType='order') where c.customerName = @name.
query = "SELECT '(placeholder)' AS id, '(placeholder)' AS date, 0 AS totalAmount"

results = list(embed_orders.query_items(
    query=query,
    parameters=[{"name": "@name", "value": customer_name}],
    partition_key=PK_VALUE,
))
ru = last_ru(embed_orders)

for o in results:
    print(f"  - {o['id']} on {o['date'][:10]}   ${o['totalAmount']:.2f}")

print(f"\n  {len(results)} order(s) for '{customer_name}', {ru:.2f} RU")

## Step 5: Hot-partition seed (provisioned account)

The remaining three cells revisit partition-key choice on the **provisioned** account. `OrdersHot` is keyed on `/orderDate` — a real-world anti-pattern where a system partitions by date and then every write "right now" lands on the same logical partition. Seed 100 orders, all with today's date, then run `SELECT DISTINCT VALUE c.orderDate FROM c` and observe **one** distinct partition-key value across the whole container.

In [ ]:
PARTITION_DEMO_COUNT = 100
HOT_CONTAINER_NAME = "OrdersHot"
COMPOSITE_CONTAINER_NAME = "OrdersComposite"

prov_db = ensure_provisioned_client()
hot_container = prov_db.get_container_client(HOT_CONTAINER_NAME)

today = datetime.now(timezone.utc).strftime("%Y-%m-%d")

for i in range(PARTITION_DEMO_COUNT):
    order = {
        "id": f"order_{i}",
        "customerId": f"CUST_{i % 50:03d}",
        "orderDate": today,
        "total": round(10 + (i * 3.33), 2),
    }
    hot_container.upsert_item(body=order)

distinct = list(hot_container.query_items(
    query="SELECT DISTINCT VALUE c.orderDate FROM c",
    enable_cross_partition_query=True,
))
print(f"  Seeded {PARTITION_DEMO_COUNT} orders. Distinct /orderDate values across the container: {len(distinct)}")
for v in distinct:
    print(f"    - {v}")

## Step 6: Composite key — inspect and re-seed (STUDENT EXERCISE)

`OrdersComposite` is keyed on `/partitionKey`. The lab uses a **synthetic composite key**: each document should write `customerId#orderDate` into `/partitionKey` so writes spread across ~50 logical partitions (one per customer for today) instead of the single one we got in Step 5.

**Your job**: replace the placeholder `pk` value with the composite of `customer_id` and `today`:

```python
pk = f"{customer_id}#{today}"
```

Run the cell. The distinct-value query should return **~50** partition-key values.

In [ ]:
prov_db = ensure_provisioned_client()
composite_container = prov_db.get_container_client(COMPOSITE_CONTAINER_NAME)

props = composite_container.read()
pk_paths = props["partitionKey"]["paths"]
print(f"  Container '{COMPOSITE_CONTAINER_NAME}' partition key paths: {pk_paths}")

for i in range(PARTITION_DEMO_COUNT):
    customer_id = f"CUST_{i % 50:03d}"

    # STUDENT EXERCISE: replace the placeholder below with a composite of customer_id
    # and today so writes spread across ~50 logical partitions.
    pk = "(placeholder)"

    composite_container.upsert_item(body={
        "id": f"order_{i}",
        "customerId": customer_id,
        "orderDate": today,
        "partitionKey": pk,
        "total": round(10 + (i * 3.33), 2),
    })

distinct = list(composite_container.query_items(
    query="SELECT DISTINCT VALUE c.partitionKey FROM c",
    enable_cross_partition_query=True,
))
print(f"  Seeded {PARTITION_DEMO_COUNT} orders. Distinct /partitionKey values across the container: {len(distinct)}")

## Step 7: Distribution summary

`OrdersHot` ended up with 1 partition-key value; `OrdersComposite` with ~50. At a small scale like 100 docs Cosmos won't have split into multiple physical partitions, so the portal heat map won't show a dramatic split — but the **logical** partition distribution is the lesson, and that's what the distinct-value query measures.

If you want to see what this looks like at production scale, open **Azure Portal > Cosmos DB > Monitoring > Insights > Throughput > Normalized RU Consumption (Max) Heat Map By PartitionKeyRangeId**. Above ~10k RU/s with multiple physical partitions, the hot container spikes one PartitionKeyRangeId while the composite stays flat.

In [ ]:
print(f"  '{HOT_CONTAINER_NAME}':       1 distinct partition-key value — all writes pinned to today's date.")
print(f"  '{COMPOSITE_CONTAINER_NAME}': ~50 distinct partition-key values — one per (customer, day).")
print()
print("At production scale (>10k RU/s with multiple physical partitions) this distribution")
print("shows up directly in Azure Monitor:")
print("  Cosmos DB > Monitoring > Insights > Throughput > 'Normalized RU Consumption (Max)'")
print("  By PartitionKeyRangeId — hot keys spike one bar, composite stays flat.")